# LLM Forecasting Replication — Step-by-Step Pipeline

Each cell is one pipeline step. Each step reads from the previous step's JSON and skips work already done.

**Pipeline:**
1. Load questions
2. Generate search queries → `queries.json`
3. Retrieve articles → `articles.json`
4. Rank + summarise → `summaries.json`
5. Reasoning + predictions → `predictions.json`

## 0. Configuration

In [1]:
# ── How many questions to evaluate ──────────────────────────────
NUM_QUESTIONS = 2

# ── Model ───────────────────────────────────────────────────────
MODEL_NAME  = "llama-3.3-70b-versatile"
TEMPERATURE = 1.0

# ── Retrieval ───────────────────────────────────────────────────
NUM_RETRIEVAL_DATES    = 5
NUM_SEARCH_QUERIES     = 3
NUM_ARTICLES_PER_QUERY = 10
TOP_K_ARTICLES         = 15
RELEVANCE_THRESHOLD    = 4

# ── Paths ────────────────────────────────────────────────────────
import os
DATA_PATH   = "../../data/test.json"          
OUTPUT_DIR  = "results"
QUERIES_FILE     = os.path.join(OUTPUT_DIR, "queries.json")
ARTICLES_FILE    = os.path.join(OUTPUT_DIR, "articles.json")
SUMMARIES_FILE   = os.path.join(OUTPUT_DIR, "summaries.json")
PREDICTIONS_FILE = os.path.join(OUTPUT_DIR, "predictions.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config OK — output dir:", OUTPUT_DIR)

Config OK — output dir: results


## 1. Imports & Helpers

In [2]:
%load_ext autoreload
%autoreload 2

import sys, json, math, logging
from datetime import datetime, timedelta
import numpy as np

REPO_ROOT = os.path.abspath("../../llm_forecasting")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from config.constants import PROMPT_DICT, DEFAULT_RETRIEVAL_CONFIG
from prompts.base_reasoning import (
    BINARY_SCRATCH_PAD_PROMPT_NEW_1,
    BINARY_SCRATCH_PAD_PROMPT_NEW_2,
    BINARY_SCRATCH_PAD_PROMPT_NEW_3,
)
import ranking
import summarize
import model_eval
import information_retrieval
from utils import time_utils, string_utils

from googlenewsdecoder import gnewsdecoder

logging.basicConfig(level=logging.INFO)

# ── Retrieval config ─────────────────────────────────────────────
RETRIEVAL_CONFIG = {
    **DEFAULT_RETRIEVAL_CONFIG,
    "SEARCH_QUERY_MODEL_NAME":     MODEL_NAME,
    "SUMMARIZATION_MODEL_NAME":    MODEL_NAME,
    "RANKING_MODEL_NAME":          MODEL_NAME,
    "NUM_SEARCH_QUERY_KEYWORDS":   NUM_SEARCH_QUERIES,
    "NUM_ARTICLES_PER_QUERY":      NUM_ARTICLES_PER_QUERY,
    "NUM_SUMMARIES_THRESHOLD":     TOP_K_ARTICLES,
    "RANKING_RELEVANCE_THRESHOLD": RELEVANCE_THRESHOLD,
    "SEARCH_QUERY_PROMPT_TEMPLATES": [
        PROMPT_DICT["search_query"]["0"],
        PROMPT_DICT["search_query"]["1"],
    ],
    "SUMMARIZATION_PROMPT_TEMPLATE": PROMPT_DICT["summarization"]["9"],
    "RANKING_PROMPT_TEMPLATE":       PROMPT_DICT["ranking"]["0"],
}

REASONING_PROMPTS = [
    BINARY_SCRATCH_PAD_PROMPT_NEW_1,
    BINARY_SCRATCH_PAD_PROMPT_NEW_2,
    BINARY_SCRATCH_PAD_PROMPT_NEW_3,
]

# ── Utility functions ────────────────────────────────────────────
def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return []

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

def get_retrieval_dates(date_begin, date_close, date_resolve, num_retrievals=5):
    dates = []
    for i in range(1, num_retrievals + 1):
        d = time_utils.get_retrieval_date(
            retrieval_index=i,
            num_retrievals=num_retrievals,
            date_begin=date_begin,
            date_close=date_close,
            resolve_date=date_resolve,
        )
        if d is not None:
            dates.append(d)
    return dates

def get_crowd_prediction_at_date(retrieval_date, community_predictions):
    preds = [p for p in community_predictions
             if time_utils.is_more_recent(p[0], retrieval_date, or_equal_to=True)]
    if not preds:
        preds = community_predictions
    closest = time_utils.find_pred_with_closest_date(retrieval_date, preds)
    return closest[1] if closest else None

def brier_score(prediction, resolution):
    return (prediction - resolution) ** 2

def article_to_dict(article):
    """Serialize a newspaper4k Article object to a plain dict."""
    return {
        "url":            getattr(article, "url", None),
        "canonical_link": getattr(article, "canonical_link", None),
        "title":          getattr(article, "title", None),
        "text":           getattr(article, "text_cleaned", None) or getattr(article, "text", None),
        "publish_date":   str(getattr(article, "publish_date", None)),
        "source":         getattr(article, "source_url", None),
        "search_term":    getattr(article, "search_term", None),
        "relevance_rating": getattr(article, "relevance_rating", None),
        "summary":        getattr(article, "summary", None),
    }

print("Imports & helpers OK")

/opt/miniconda3/envs/myenv/lib/python3.11/site-packages/newspaper/parsers.py:19: UserWarning: nltk is not installed. Some NLP features will be unavailable. Install it with: pip install 'newspaper4k[nlp]'
  from . import text as txt
INFO:newspaper.network:Using requests library for http requests (alternative cloudscraper library is recommended for bypassing Cloudflare protection)
/opt/miniconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sonia/Documents/llm_forecasting/llm_forecasting/model_eval.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai

Imports & helpers OK


## 2. Load Questions

In [3]:
with open(DATA_PATH) as f:
    all_questions = json.load(f)

questions_to_run = all_questions[:NUM_QUESTIONS]
print(f"Loaded {len(all_questions)} questions. Running on {len(questions_to_run)}.")
for i, q in enumerate(questions_to_run):
    print(f"  [{i}] {q['question']}")

Loaded 914 questions. Running on 2.
  [0] Will Ukraine retake Polohy by the 1st of October, 2023?
  [1] Will the EU Parliament endorse a negotiating mandate for the AI Act before June 16, 2023?


## 3. Step 1 — Generate Search Queries → `queries.json`

For each question × retrieval date, generate GNews search queries via LLM.
Skips any (question, retrieval_date) pair already saved.

In [4]:
queries_data = load_json(QUERIES_FILE)  # list of question-level dicts

# Build a lookup: question_text -> existing entry
queries_lookup = {entry["question"]: entry for entry in queries_data}

for q in questions_to_run:
    question = q["question"]
    print(f"\nQuestion: {question[:80]}")

    # Get or create the entry for this question
    if question not in queries_lookup:
        queries_lookup[question] = {
            "question":        question,
            "resolution":      float(q["resolution"]),
            "date_begin":      q["date_begin"],
            "date_close":      q["date_close"],
            "date_resolve_at": q["date_resolve_at"],
            "retrieval_dates": get_retrieval_dates(
                q["date_begin"], q["date_close"], q["date_resolve_at"], NUM_RETRIEVAL_DATES
            ),
            "retrieval_date_queries": {}   # retrieval_date -> list of queries
        }

    entry = queries_lookup[question]
    existing_dates = entry["retrieval_date_queries"]

    for retrieval_date in entry["retrieval_dates"]:
        if retrieval_date in existing_dates:
            print(f"  [{retrieval_date}] Already have queries — skipping.")
            continue

        print(f"  [{retrieval_date}] Generating queries...")
        date_range = [q["date_begin"], retrieval_date]
        try:
            (
                search_queries_list_nc,
                search_queries_list_gnews,
                _, _, _, _,
            ) = await information_retrieval.get_search_queries_for_all_sources(
                RETRIEVAL_CONFIG["SEARCH_QUERY_PROMPT_TEMPLATES"],
                RETRIEVAL_CONFIG["NUM_SEARCH_QUERY_KEYWORDS"],
                date_range,
                question,
                background_info=q["background"],
                resolution_criteria=q["resolution_criteria"],
            )
            # Flatten and deduplicate GNews queries only
            from utils import utils as utils_module
            queries_gnews = list(set(utils_module.flatten_list(search_queries_list_gnews)))
            existing_dates[retrieval_date] = queries_gnews
            print(f"    Generated {len(queries_gnews)} queries: {queries_gnews}")
        except Exception as e:
            print(f"    [ERROR] {e}")
            existing_dates[retrieval_date] = []

        # Save after every retrieval date
        queries_data = list(queries_lookup.values())
        save_json(QUERIES_FILE, queries_data)

print(f"\nDone. Saved to {QUERIES_FILE}")


Question: Will Ukraine retake Polohy by the 1st of October, 2023?
  [2023-06-03] Already have queries — skipping.
  [2023-06-07] Already have queries — skipping.
  [2023-06-18] Already have queries — skipping.
  [2023-07-17] Already have queries — skipping.
  [2023-09-29] Already have queries — skipping.

Question: Will the EU Parliament endorse a negotiating mandate for the AI Act before June 
  [2023-06-03] Already have queries — skipping.
  [2023-06-04] Already have queries — skipping.
  [2023-06-06] Already have queries — skipping.
  [2023-06-10] Already have queries — skipping.

Done. Saved to results/queries.json


## 4. Step 2 — Retrieve Articles → `articles.json`

For each query, fetch articles from GNews and extract full text.
Skips any query already saved. Reports missing full text.

In [5]:
# ── Flag: set to True to rerun queries already saved ────────────
FORCE_RERUN_ARTICLES = True

articles_data = load_json(ARTICLES_FILE)
articles_lookup = {entry["question"]: entry for entry in articles_data}

for q_entry in queries_data:
    question = q_entry["question"]
    print(f"\nQuestion: {question[:80]}")

    if question not in articles_lookup:
        articles_lookup[question] = {
            "question":    question,
            "resolution":  q_entry["resolution"],
            "date_begin":  q_entry["date_begin"],
            "date_close":  q_entry["date_close"],
            "retrieval_dates": q_entry["retrieval_dates"],
            "retrieval_date_articles": {}
        }

    art_entry = articles_lookup[question]
    date_art  = art_entry["retrieval_date_articles"]

    # Per-question registry: decoded_url -> article dict (shared across all dates/queries)
    url_registry = {}
    # Populate registry from already-saved articles (for FORCE_RERUN or resume cases)
    for date_articles in date_art.values():
        for query_articles in date_articles.values():
            for a in query_articles:
                if a.get("url"):
                    url_registry[a["url"]] = a

    for retrieval_date, queries in q_entry["retrieval_date_queries"].items():
        if retrieval_date not in date_art:
            date_art[retrieval_date] = {}

        date_range = [q_entry["date_begin"], retrieval_date]

        for query in queries:
            if not FORCE_RERUN_ARTICLES and query in date_art[retrieval_date]:
                print(f"  [{retrieval_date}] '{query[:50]}' — already fetched, skipping.")
                continue

            print(f"  [{retrieval_date}] Fetching: '{query[:60]}'")
            try:
                # get_gnews_articles returns a list of dicts
                raw_dicts = information_retrieval.get_gnews_articles(
                    [query], date_range, max_results=NUM_ARTICLES_PER_QUERY
                )
                # flatten in case it returns list of lists
                if raw_dicts and isinstance(raw_dicts[0], list):
                    raw_dicts = [a for sublist in raw_dicts for a in sublist]

                print(f"    Raw articles: {len(raw_dicts)}")

                # Scrape full text for each article dict directly
                saved = []
                for art_dict in raw_dicts:
                    url = art_dict.get("url", "")
                    # Decode the Google News URL to the real article URL
                    # Decode the Google News URL to the real article URL
                    decoded_url = url
                    try:
                        result = gnewsdecoder(url, interval=1)
                        if result.get("status"):
                            decoded_url = result["decoded_url"]
                    except Exception:
                        pass

                    if decoded_url in url_registry:
                        # Already scraped — just increment times_appearing
                        url_registry[decoded_url]["times_appearing"] += 1
                        entry = url_registry[decoded_url]
                        print(f"      Duplicate: '{decoded_url[:60]}' (times_appearing={entry['times_appearing']})")
                    else:
                        # New article — scrape and register
                        entry = {
                            "title":           art_dict.get("title"),
                            "description":     art_dict.get("description"),
                            "url":             decoded_url,
                            "google_news_url": url,
                            "publish_date":    art_dict.get("published date"),
                            "source":          art_dict.get("publisher", {}).get("title"),
                            "source_url":      art_dict.get("publisher", {}).get("href"),
                            "search_term":     art_dict.get("search_term"),
                            "text":            None,
                            "times_appearing": 1,
                        }
                        try:
                            article_obj = information_retrieval.retrieve_webpage_text(decoded_url, retrieval_date)
                            if article_obj:
                                entry["text"] = getattr(article_obj, "text_cleaned", None) or getattr(article_obj, "text", None)
                        except Exception:
                            pass
                        url_registry[decoded_url] = entry

                    saved.append(entry)

                with_text = len([a for a in saved if a["text"]])
                missing   = len(saved) - with_text
                print(f"    Saved {len(saved)} articles, {with_text} with full text, {missing} without.")

                date_art[retrieval_date][query] = saved

            except Exception as e:
                print(f"    [ERROR] {e}")
                date_art[retrieval_date][query] = []

            # Save after every query
            articles_data = list(articles_lookup.values())
            save_json(ARTICLES_FILE, articles_data)

print(f"\nDone. Saved to {ARTICLES_FILE}")


Question: Will advanced LIGO announce discovery of gravitational waves by Jan. 31 2016?
  [2015-10-04] Fetching: 'advanced LIGO latest results 2015'


INFO:information_retrieval:Retrieved 0 articles for advanced LIGO latest results 2015 via GNews.


    Raw articles: 0
    Saved 0 articles, 0 with full text, 0 without.
  [2015-10-04] Fetching: 'advanced LIGO observation runs schedule'


INFO:information_retrieval:Retrieved 0 articles for advanced LIGO observation runs schedule via GNews.


    Raw articles: 0
    Saved 0 articles, 0 with full text, 0 without.
  [2015-10-04] Fetching: 'advanced LIGO announcement plans 2016'


...

    Raw articles: 3
      Duplicate: 'https://www.europarl.europa.eu/topics/en/article/20230601STO' (times_appearing=3)


INFO:information_retrieval:Good article: publish date 2023-06-06 earlier than the end date 2023-06-10.
INFO:information_retrieval:Good article: publish date 2023-06-09 earlier than the end date 2023-06-10.


    Saved 3 articles, 2 with full text, 1 without.
  [2023-06-10] Fetching: 'EU Parliament AI Act negotiating mandate endorsement'


INFO:information_retrieval:Retrieved 1 articles for EU Parliament AI Act negotiating mandate endorsement via GNews.


    Raw articles: 1
      Duplicate: 'https://www.europarl.europa.eu/topics/en/article/20230601STO' (times_appearing=4)
    Saved 1 articles, 0 with full text, 1 without.

Done. Saved to results/articles.json


## 5. Step 3 — Rank & Summarise → `summaries.json`

Pool all articles for each retrieval date, run LLM relevance ranking,
keep top K, summarise. Skips question+date combos already saved.

In [ ]:
FORCE_RERUN_SUMMARIES = True

articles_data  = load_json(ARTICLES_FILE)
summaries_data = load_json(SUMMARIES_FILE)

summaries_lookup = {entry["question"]: entry for entry in summaries_data}

# We need the original question metadata for prompts
q_meta = {q["question"]: q for q in questions_to_run}

for art_entry in articles_data:
    question = art_entry["question"]
    print(f"\nQuestion: {question[:80]}")

    if question not in summaries_lookup:
        summaries_lookup[question] = {
            "question":   question,
            "resolution": art_entry["resolution"],
            "date_begin": art_entry["date_begin"],
            "date_close": art_entry["date_close"],
            "retrieval_dates": art_entry["retrieval_dates"],
            "retrieval_date_summaries": {}  # retrieval_date -> summary string + metadata
        }

    sum_entry  = summaries_lookup[question]
    date_sums  = sum_entry["retrieval_date_summaries"]
    q_info     = q_meta.get(question, {})

    for retrieval_date, query_articles in art_entry["retrieval_date_articles"].items():
        if not FORCE_RERUN_SUMMARIES and retrieval_date in date_sums:
            print(f"  [{retrieval_date}] Already summarised — skipping.")
            continue

        print(f"  [{retrieval_date}] Ranking and summarising...")

        # Pool all articles across queries and deduplicate by URL
        seen_urls = set()
        pooled = []
        for articles_list in query_articles.values():
            for art in articles_list:
                url = art.get("url") or art.get("canonical_link")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    pooled.append(art)

        print(f"    Pooled {len(pooled)} unique articles.")

        if len(pooled) == 0:
            print("    No articles to rank — saving empty summary.")
            date_sums[retrieval_date] = {
                "num_articles_pooled": 0,
                "num_articles_ranked": 0,
                "ranked_articles":     [],
                "summary":             "No articles were retrieved for this question."
            }
            save_json(SUMMARIES_FILE, list(summaries_lookup.values()))
            continue

        # Convert dicts back to objects for ranking (reconstruct minimal objects)
        # The ranking function needs objects with .text_cleaned, .title etc.
        # Easiest: use the repo's own Article-like objects via newspaper4k
        class SimpleArticle:
            """Minimal article object compatible with the ranking function."""
            def __init__(self, d):
                self.url          = d.get("url") or ""
                self.canonical_link = d.get("canonical_link") or d.get("url") or ""
                self.title        = d.get("title") or ""
                self.text         = d.get("text") or ""
                self.text_cleaned = d.get("text") or ""
                self.publish_date = d.get("publish_date")
                self.source_url   = d.get("source")
                self.search_term  = d.get("search_term") or ""
                self.relevance_rating   = None
                self.relevance_response = None
                self.summary      = None

        article_objects = [SimpleArticle(d) for d in pooled]
        print(f"    Created {len(article_objects)} article objects.")

        # Run LLM ranking — get ALL articles back with their scores first
        try:
            # Temporarily set threshold to 0 to get all scores before filtering
            all_rated = await ranking.rank_articles(
                article_objects,
                method=RETRIEVAL_CONFIG["RANKING_METHOD"],
                method_llm=RETRIEVAL_CONFIG["RANKING_METHOD_LLM"],
                prompt_template=RETRIEVAL_CONFIG["RANKING_PROMPT_TEMPLATE"],
                question=question,
                dates=[art_entry["date_begin"], retrieval_date],
                resolution_criteria=q_info.get("resolution_criteria", ""),
                background=q_info.get("background", ""),
                sort_by=RETRIEVAL_CONFIG["SORT_BY"],
                relevance_rating_threshold=0,  # get everything, filter manually below
                model_name=RETRIEVAL_CONFIG["RANKING_MODEL_NAME"],
                temperature=RETRIEVAL_CONFIG["RANKING_TEMPERATURE"],
            )
        except Exception as e:
            print(f"    [ERROR] Ranking failed: {e}")
            all_rated = []

        # Log all scores so we can see what's happening
        print(f"    Scores for all {len(all_rated)} articles:")
        for a in all_rated:
            score = getattr(a, "relevance_rating", None)
            title = getattr(a, "title", "")[:60]
            print(f"      [{score}] {title}")

        # Now filter by threshold
        ranked = [a for a in all_rated
                  if getattr(a, "relevance_rating", 0) is not None
                  and getattr(a, "relevance_rating", 0) >= RETRIEVAL_CONFIG["RANKING_RELEVANCE_THRESHOLD"]]
        ranked = ranked[:TOP_K_ARTICLES]
        print(f"    {len(ranked)} articles survived ranking (threshold={RETRIEVAL_CONFIG['RANKING_RELEVANCE_THRESHOLD']}).")

        # Summarise
        if len(ranked) > 0:
            prompt = RETRIEVAL_CONFIG["SUMMARIZATION_PROMPT_TEMPLATE"][0].format(
                question=question,
                background=q_info.get("background", ""),
                article="{article}",
            )
            try:
                await summarize.summarize_articles(ranked, prompt=prompt, update_object=True,
                    temperature=RETRIEVAL_CONFIG["SUMMARIZATION_TEMPERATURE"],
                    model_name=RETRIEVAL_CONFIG["SUMMARIZATION_MODEL_NAME"])
            except Exception as e:
                print(f"    [ERROR] Summarisation failed: {e}")

        combined_summary = summarize.concat_summaries(ranked)

        # Build detailed per-article ranking info
        def article_to_dict_with_ranking(a):
            d = article_to_dict(a)
            d["relevance_rating"]       = getattr(a, "relevance_rating", None)
            d["relevance_llm_response"] = getattr(a, "relevance_response", None) or getattr(a, "rating_response", None)
            d["summary"]                = getattr(a, "summary", None)
            return d

        date_sums[retrieval_date] = {
            "num_articles_pooled":  len(pooled),
            "num_articles_rated":   len(all_rated),
            "num_articles_ranked":  len(ranked),
            "all_rated_articles":   [article_to_dict_with_ranking(a) for a in all_rated],
            "ranked_articles":      [article_to_dict_with_ranking(a) for a in ranked],
            "summary":              combined_summary,
        }
        save_json(SUMMARIES_FILE, list(summaries_lookup.values()))
        print(f"    Summary saved ({len(combined_summary)} chars).")

print(f"\nDone. Saved to {SUMMARIES_FILE}")

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 23.000000 seconds



Question: Will advanced LIGO announce discovery of gravitational waves by Jan. 31 2016?
  [2015-10-04] Ranking and summarising...
    Pooled 0 unique articles.
    No articles to rank — saving empty summary.
  [2015-10-07] Ranking and summarising...
    Pooled 2 unique articles.
    Created 2 article objects.


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:ranking:Article LIGO's SURF Students Look for the Perfect Wave - Caltech gets rating: 2.0
INFO:ranking:Article Using general relativity to magnify the cosmos - Science News gets rating: 2.0
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds


    [ERROR] Ranking failed: 'str' object has no attribute 'date'
    Scores for all 0 articles:
    0 articles survived ranking (threshold=4).
    Summary saved (54 chars).
  [2015-10-15] Ranking and summarising...
    Pooled 1 unique articles.
    Created 1 article objects.


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:ranking:Article LIGO's SURF Students Look for the Perfect Wave - Caltech gets rating: 2.0
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 41.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 41.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 41.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 41.000000 seconds
INFO:httpx:HTTP Request: POST

...

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 39.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST h

    [ERROR] Ranking failed: 'str' object has no attribute 'date'
    Scores for all 0 articles:
    0 articles survived ranking (threshold=4).
    Summary saved (54 chars).
  [2023-06-18] Ranking and summarising...
    Pooled 13 unique articles.
    Created 13 article objects.


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 1.000000 seconds
INFO:httpx:HTTP Request: POST http

    [ERROR] Ranking failed: 'str' object has no attribute 'date'
    Scores for all 0 articles:
    0 articles survived ranking (threshold=4).
    Summary saved (54 chars).
  [2023-07-17] Ranking and summarising...
    Pooled 22 unique articles.
    Created 22 article objects.


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 6.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 45.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 44.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 5.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 5.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POS

In [10]:
# DEBUG — call rank_articles on a small set and inspect the result
test_q = "Will Ukraine retake Polohy by the 1st of October, 2023?"
test_art_entry = next(e for e in articles_data if e["question"] == test_q)
test_date = "2023-07-17"  # pick the date with 22 pooled articles

# Reconstruct article objects same as Step 3
from newspaper import Article
test_pooled = []
seen = set()
for articles_list in test_art_entry["retrieval_date_articles"][test_date].values():
    for art in articles_list:
        url = art.get("url")
        if url and url not in seen:
            seen.add(url)
            test_pooled.append(art)

print(f"Pooled: {len(test_pooled)} articles")

article_objects = []
for art_dict in test_pooled:
    try:
        a = Article(url=art_dict.get("url") or "http://unknown")
        a.set_text(art_dict.get("text") or "")
        a.title = art_dict.get("title") or ""
        a.text_cleaned = art_dict.get("text") or ""
        a.publish_date = art_dict.get("publish_date")
        a.source_url   = art_dict.get("source")
        a.search_term  = art_dict.get("search_term") or ""
        a.canonical_link = art_dict.get("url") or ""
        article_objects.append(a)
    except Exception as e:
        print(f"  Object creation error: {e}")

print(f"Article objects created: {len(article_objects)}")
print(f"First article text length: {len(article_objects[0].text_cleaned) if article_objects else 0}")
print(f"First article title: {article_objects[0].title if article_objects else 'N/A'}")

# Try ranking just 2 articles
result = await ranking.rank_articles(
    article_objects[:2],
    method=RETRIEVAL_CONFIG["RANKING_METHOD"],
    method_llm=RETRIEVAL_CONFIG["RANKING_METHOD_LLM"],
    prompt_template=RETRIEVAL_CONFIG["RANKING_PROMPT_TEMPLATE"],
    question=test_q,
    dates=[test_art_entry["date_begin"], test_date],
    resolution_criteria="",
    background="",
    sort_by=RETRIEVAL_CONFIG["SORT_BY"],
    relevance_rating_threshold=0,
    model_name=RETRIEVAL_CONFIG["RANKING_MODEL_NAME"],
    temperature=RETRIEVAL_CONFIG["RANKING_TEMPERATURE"],
)
print(f"Ranking result: {len(result)} articles")
if result:
    print(f"First result attrs: {vars(result[0]).keys()}")
    print(f"Relevance rating: {getattr(result[0], 'relevance_rating', 'NOT FOUND')}")

Pooled: 22 articles
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'
  Object creation error: 'Article' object has no attribute 'set_text'


## 6. Step 4 — Reasoning & Predictions → `predictions.json`

For each question × retrieval date, run 3 scratchpad prompts, average predictions,
compute Brier score vs crowd. Skips question+date combos already predicted.

In [ ]:
summaries_data   = load_json(SUMMARIES_FILE)
predictions_data = load_json(PREDICTIONS_FILE)

predictions_lookup = {entry["question"]: entry for entry in predictions_data}

# Community predictions lookup from original dataset
community_preds_lookup = {q["question"]: json.loads(q["community_predictions"]) for q in questions_to_run}

for sum_entry in summaries_data:
    question   = sum_entry["question"]
    resolution = float(sum_entry["resolution"])
    print(f"\nQuestion: {question[:80]}")

    if question not in predictions_lookup:
        predictions_lookup[question] = {
            "question":    question,
            "resolution":  resolution,
            "date_begin":  sum_entry["date_begin"],
            "date_close":  sum_entry["date_close"],
            "retrieval_dates": sum_entry["retrieval_dates"],
            "retrieval_date_predictions": {}  # retrieval_date -> prediction details
        }

    pred_entry  = predictions_lookup[question]
    date_preds  = pred_entry["retrieval_date_predictions"]
    community_preds = community_preds_lookup.get(question, [])

    for retrieval_date, sum_info in sum_entry["retrieval_date_summaries"].items():
        if retrieval_date in date_preds:
            print(f"  [{retrieval_date}] Already predicted — skipping.")
            continue

        print(f"  [{retrieval_date}] Running 3 reasoning prompts...")
        all_summaries   = sum_info["summary"]
        today_to_close  = [retrieval_date, sum_entry["date_close"]]
        q_info          = {q["question"]: q for q in questions_to_run}.get(question, {})

        base_reasonings  = []
        base_predictions = []
        full_prompts     = []

        for p_idx, prompt_template in enumerate(REASONING_PROMPTS):
            print(f"    Prompt {p_idx+1}/3 ...")
            try:
                reasonings, prompts = await model_eval.async_make_forecast(
                    question=question,
                    background_info=q_info.get("background", ""),
                    resolution_criteria=q_info.get("resolution_criteria", ""),
                    dates=today_to_close,
                    retrieved_info=all_summaries,
                    reasoning_prompt_templates=[prompt_template],
                    model_name=MODEL_NAME,
                    temperature=TEMPERATURE,
                    return_prompt=True,
                )
                reasoning  = reasonings[0]
                prediction = string_utils.extract_prediction(reasoning, answer_type="probability")
                base_reasonings.append(reasoning)
                base_predictions.append(prediction)
                full_prompts.append(prompts[0])
                print(f"      Prediction: {prediction}")
            except Exception as e:
                print(f"      [ERROR] {e}")
                base_reasonings.append(None)
                base_predictions.append(None)
                full_prompts.append(None)

        valid_preds     = [p for p in base_predictions if p is not None]
        mean_prediction = float(np.mean(valid_preds)) if valid_preds else None
        crowd_pred      = get_crowd_prediction_at_date(retrieval_date, community_preds)
        llm_brier       = brier_score(mean_prediction, resolution) if mean_prediction is not None else None
        crowd_brier     = brier_score(crowd_pred, resolution) if crowd_pred is not None else None

        print(f"    Mean prediction: {mean_prediction}")
        print(f"    Crowd prediction: {crowd_pred}")
        print(f"    LLM Brier: {f'{llm_brier:.4f}' if llm_brier is not None else 'N/A'}  "
              f"Crowd Brier: {f'{crowd_brier:.4f}' if crowd_brier is not None else 'N/A'}")

        date_preds[retrieval_date] = {
            "base_reasonings":  base_reasonings,
            "full_prompts":     full_prompts,
            "base_predictions": base_predictions,
            "mean_prediction":  mean_prediction,
            "crowd_prediction": crowd_pred,
            "llm_brier_score":  llm_brier,
            "crowd_brier_score":crowd_brier,
        }

        # Save after every retrieval date
        save_json(PREDICTIONS_FILE, list(predictions_lookup.values()))

    # Aggregate across dates
    llm_briers   = [v["llm_brier_score"]   for v in date_preds.values() if v.get("llm_brier_score")   is not None]
    crowd_briers = [v["crowd_brier_score"]  for v in date_preds.values() if v.get("crowd_brier_score") is not None]
    pred_entry["mean_llm_brier"]   = float(np.mean(llm_briers))   if llm_briers   else None
    pred_entry["mean_crowd_brier"] = float(np.mean(crowd_briers)) if crowd_briers else None
    save_json(PREDICTIONS_FILE, list(predictions_lookup.values()))

    print(f"  ► Mean LLM Brier:   {pred_entry['mean_llm_brier']}")
    print(f"  ► Mean Crowd Brier: {pred_entry['mean_crowd_brier']}")

print(f"\nDone. Saved to {PREDICTIONS_FILE}")

## 7. Summary

In [ ]:
predictions_data = load_json(PREDICTIONS_FILE)

print(f"{'Question':<60} {'LLM Brier':>10} {'Crowd Brier':>12}")
print("-" * 84)
all_llm, all_crowd = [], []
for r in predictions_data:
    q_short = r["question"][:57] + "..." if len(r["question"]) > 60 else r["question"]
    llm_b   = f"{r['mean_llm_brier']:.4f}"   if r.get("mean_llm_brier")   is not None else "N/A"
    crowd_b = f"{r['mean_crowd_brier']:.4f}" if r.get("mean_crowd_brier") is not None else "N/A"
    print(f"{q_short:<60} {llm_b:>10} {crowd_b:>12}")
    if r.get("mean_llm_brier")   is not None: all_llm.append(r["mean_llm_brier"])
    if r.get("mean_crowd_brier") is not None: all_crowd.append(r["mean_crowd_brier"])

if all_llm:
    print("-" * 84)
    print(f"{'OVERALL MEAN':<60} {np.mean(all_llm):>10.4f} {np.mean(all_crowd):>12.4f}")